# Tugas Besar ML — Part 3
## Ekspor & unduh model `.pth` dari Google Colab

Notebook ini dipakai untuk demonstrasi **Part 3**: model hasil training di Colab diekspor menjadi satu file `model_cuaca_kemayoran.pth`, lalu diunduh ke komputer untuk dipakai aplikasi web.

**Urutan pengerjaan**
1. Jalankan `notebooks/01-training.ipynb` sampai sel terakhir (menghasilkan `model.pkl`, `features.pkl`, `metadata.json`).
2. Jalankan notebook ini sampai sel **download**.
3. Screenshot proses ekspor dan dialog download, simpan ke `docs/screenshots/`.
4. Taruh file `.pth` hasil download di folder `model/`.

> File `.pth` disimpan dengan `torch.save` bila torch tersedia (Colab), dengan fallback pickle. Isinya **bundle deployment**: pipeline scikit-learn + fitur + threshold + metadata — bukan state_dict PyTorch. API tetap bisa membacanya tanpa menginstal `torch`.


## 1. Siapkan artefak model

Jika notebook ini dibuka di Colab yang sama dengan `01-training.ipynb`, artefak sudah ada di `/content/model`. Jika belum, unggah `model.pkl`, `features.pkl`, dan `metadata.json`.

In [ ]:
from pathlib import Path
import hashlib
import zipfile
import json
import pickle
from datetime import datetime, timezone

import joblib
import pandas as pd

CANDIDATES = [Path('/content/model'), Path('model'), Path('../model')]
MODEL_DIR = next((p for p in CANDIDATES if (p / 'model.pkl').exists()), CANDIDATES[0])

if not (MODEL_DIR / 'model.pkl').exists():
    MODEL_DIR = Path('/content/model')
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files as colab_files  # type: ignore
        print('Unggah model.pkl, features.pkl, metadata.json:')
        uploaded = colab_files.upload()
        for name, data in uploaded.items():
            (MODEL_DIR / name).write_bytes(data)
    except ImportError:
        raise FileNotFoundError('Letakkan model.pkl di folder model/ terlebih dahulu.')

print('Folder artefak :', MODEL_DIR.resolve())
print('Isi folder     :', sorted(p.name for p in MODEL_DIR.iterdir()))


## 2. Bentuk bundle `.pth`

Satu file memuat semuanya supaya aplikasi cukup membaca satu artefak. Di Colab, file disimpan dengan `torch.save` sehingga benar-benar berformat `.pth` PyTorch.


In [ ]:
model = joblib.load(MODEL_DIR / 'model.pkl')
features = joblib.load(MODEL_DIR / 'features.pkl')
metadata = json.loads((MODEL_DIR / 'metadata.json').read_text())
threshold = float(metadata.get('threshold', 0.5))

# Kompatibilitas model lama (scikit-learn 1.6) di scikit-learn baru.
for _, est in getattr(model, 'steps', []):
    if type(est).__name__ == 'SimpleImputer' and not hasattr(est, '_fill_dtype'):
        est._fill_dtype = getattr(est, '_fit_dtype', float)

bundle = {
    '_format': 'sklearn-deployment-bundle',
    '_format_version': 1,
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'model': model,
    'features': list(features),
    'threshold': threshold,
    'metadata': metadata,
}

OUT = MODEL_DIR / 'model_cuaca_kemayoran.pth'

def read_bundle(path):
    """Baca .pth hasil torch.save atau bundle pickle biasa."""
    with open(path, 'rb') as f:
        head = f.read(2)
    if head == b'PK':
        try:
            import torch
            return torch.load(path, map_location='cpu', weights_only=False)
        except ImportError:
            with zipfile.ZipFile(path) as zf:
                names = [n for n in zf.namelist()
                         if n == 'data.pkl' or n.endswith('/data.pkl')]
                with zf.open(names[0]) as f:
                    return pickle.load(f)
    with open(path, 'rb') as f:
        return pickle.load(f)

try:
    import torch
    torch.save(bundle, OUT)
    bundle_format = f'torch.save (torch {torch.__version__})'
except ImportError:
    with open(OUT, 'wb') as f:
        pickle.dump(bundle, f, protocol=4)
    bundle_format = 'pickle (torch tidak terpasang)'

digest = hashlib.sha256(OUT.read_bytes()).hexdigest()
(MODEL_DIR / 'model_cuaca_kemayoran.pth.sha256').write_text(
    f'{digest}  {OUT.name}\n')

print(f'Tersimpan : {OUT}')
print(f'Format    : {bundle_format}')
print(f'Ukuran    : {OUT.stat().st_size / 1024:.1f} KB')
print(f'SHA-256   : {digest}')
print(f'Fitur     : {len(features)}')
print(f'Threshold : {threshold}')


## 3. Uji file `.pth`

Pastikan file hasil ekspor benar-benar bisa memprediksi sebelum diunduh.

In [ ]:
check = read_bundle(OUT)

sample = {
    'Tn': 24.0, 'Tx': 31.0, 'Tavg': 27.5, 'RH_avg': 85.0,
    'ss': 1.5, 'ff_x': 10.0, 'ff_avg': 4.0,
    'rain_yesterday': 35.0, 'month': 1,
}
X = pd.DataFrame([{f: float(sample[f]) for f in check['features']}])
proba = float(check['model'].predict_proba(X)[0][1])
label = 'HUJAN LEBAT' if proba >= check['threshold'] else 'TIDAK HUJAN LEBAT'

print(json.dumps({
    'prediction': label,
    'prob': round(proba, 4),
    'threshold': check['threshold'],
    'artifact': OUT.name,
}, indent=2))


## 4. Unduh model ke komputer (screenshot)

Jalankan sel ini. Browser akan menampilkan dialog download `model_cuaca_kemayoran.pth` — **screenshot dialog tersebut** untuk lampiran laporan Part 3.

In [ ]:
from google.colab import files as colab_files  # type: ignore
colab_files.download(str(OUT))

## 5. Catatan format `.pth`

File hasil ekspor memakai `torch.save` bila torch tersedia, atau pickle bila tidak. Isinya adalah **bundle objek** (pipeline scikit-learn, fitur, threshold, metadata), bukan `state_dict` jaringan saraf.

Aplikasi `api/app.py` dapat membuka keduanya: arsip `torch.save` dibaca dengan `torch.load` bila torch terpasang, atau dengan fallback `zipfile` + `pickle` tanpa torch.


In [ ]:
print('File .pth siap dipakai aplikasi. Jalankan sel download di atas.')
